In [141]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
import numpy as np
import pickle 

In [142]:
class  Convo:
    def __init__(self, window_size, n_filter,n_channel):
        self.window_size = window_size
        self.n_filter = n_filter
        self.n_channel = n_channel
        self.kernel = np.random.randn(window_size, window_size, n_channel, n_filter) * np.sqrt(2 / (window_size * window_size * n_channel))
        self.biases = np.zeros((n_filter))

        self.kernel_momentumn = np.zeros_like(self.kernel)
        self.biases_momentumn = np.zeros_like(self.biases)


    def forward(self, inputs):
        ws = self.window_size 
        lower = int(np.floor(ws/2))
        upper = int(np.ceil(ws/2))
        self.inputs = inputs 
        batch_size, W, H, channel = inputs.shape

        if channel != self.n_channel:
            raise ValueError("input channel does not match kernel channel")

        self.feature_maps = np.zeros((batch_size, W - ws + 1, H - ws + 1, self.n_filter))

        for b in range(batch_size):
            for f in range(self.n_filter):  
                for w in range(lower, W - upper +1 ):
                    for h in range(lower, H - upper + 1): 
                        self.feature_maps[b,w-lower,h-lower,f] = np.sum(inputs[b,w-lower:w+upper,h-lower:h+upper,:] * self.kernel[:,:,:,f]) + self.biases[f]
        self.output = self.feature_maps


    def backward(self,dvalues):
        self.d_biases = np.sum(dvalues,axis=(0,1,2))
        self.d_kernel = np.zeros_like(self.kernel)
        self.d_inputs = np.zeros_like(self.inputs)

        ws = self.window_size 
        lower = int(np.floor(ws/2))
        upper = int(np.ceil(ws/2))
        batch_size, W, H, channel = self.inputs.shape

        for b in range(batch_size):
            for f in range(self.n_filter):  
                for w in range(lower, W - upper +1 ):
                    for h in range(lower, H - upper + 1): 
                        grad = dvalues[b,w-lower,h-lower,f]
                        self.d_kernel[:,:,:,f] += self.inputs[b,w-lower:w+upper,h-lower:h+upper,:] * grad
                        self.d_inputs[b,w-lower:w+upper,h-lower:h+upper,:] += self.kernel[:,:,:,f] * grad


class Pooling:
    def __init__(self, pool_size=2, stride = 2, mode = 'max'):
        self.pool_size = pool_size
        self.stride = stride
        self.mode = mode

    def forward(self,inputs):
        self.inputs = inputs
        ps = self.pool_size
        s = self.stride
        batch_size, W, H, n_filter = inputs.shape
        output_W = int(np.floor((W - ps)/s)) + 1
        output_H = int(np.floor((H - ps)/s)) + 1

        self.output = np.zeros((batch_size, output_W, output_H, n_filter))

        for b in range(batch_size):
            for f in range(n_filter):  
                for w in range(output_W ):
                    for h in range(output_H): 
                        w_start = w*s
                        h_start = h*s
                        patch = inputs[b, w_start : w_start + ps, h_start : h_start + ps,f]
                        if self.mode == 'max':
                            self.output[b, w, h, f] = np.max(patch) 
                        elif self.mode == 'mean':
                            self.output[b, w, h, f] = np.mean(patch) 
                        else:
                            print('invalid pooling mode')
                            raise ValueError("please choose 'max' or 'mean'")
                        

    def backward(self,dvalues):

        inputs = self.inputs
        ps = self.pool_size
        s = self.stride
        batch_size, W, H, n_filter = inputs.shape
        output_W = int(np.floor((W - ps)/s)) + 1
        output_H = int(np.floor((H - ps)/s)) + 1
        self.d_inputs = np.zeros_like(inputs)

        for b in range(batch_size):
            for f in range(n_filter):  
                for w in range(output_W ):
                    for h in range(output_H): 
                        w_start = w*s
                        h_start = h*s
                        patch = inputs[b, w_start : w_start + ps, h_start : h_start + ps,f]

                        if self.mode == 'max':
                            mask = (patch == np.max(patch))
                            self.d_inputs[b, w_start:w_start+ps, h_start:h_start+ps, f] += mask * dvalues[b, w, h, f]

                        elif self.mode == 'mean':
                            self.d_inputs[b, w_start:w_start+ps, h_start:h_start+ps, f] += float(dvalues[b, w, h, f] / (self.pool_size**2))



class conv_optimizer:
    def __init__(self,lr = 0.001, beta = 0.9):
        self.lr = lr
        self.beta = beta

    def OptimizerGD(self,layer):
        layer.kernel -= self.lr*layer.d_kernel
        layer.biases -= self.lr*layer.d_biases

    def OptimizerGD_WithMomentumn(self, layer):
        
        layer.kernel_momentumn = self.beta * layer.kernel_momentumn - self.lr*layer.d_kernel
        layer.biases_momentumn = self.beta * layer.biases_momentumn - self.lr*layer.d_biases

        layer.kernel += layer.kernel_momentumn
        layer.biases += layer.biases_momentumn


In [143]:
class LayerDense:

    def __init__(self,n_input,n_neuron):
        self.weights = np.random.randn(n_input, n_neuron) * np.sqrt(2 / n_input)
        self.biases = np.zeros((1,n_neuron))

        self.weight_momentumn = np.zeros_like(self.weights)
        self.biases_momentumn = np.zeros_like(self.biases)

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs,self.weights) + self.biases

    def backward(self, dvalues):
        self.d_biases = np.sum(dvalues,axis=0,keepdims=True)
        self.d_weights = np.dot(self.inputs.T,dvalues)
        self.d_inputs = np.dot(dvalues,self.weights.T)


class relu:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0,inputs)

    def backward(self,dvalues):
        self.d_inputs = np.where(self.inputs <= 0, 0, dvalues)

class tanh:
    def forward(self,inputs):
        self.inputs = inputs
        self.output = (np.exp(inputs) - np.exp(-inputs)) / (np.exp(inputs) + np.exp(-inputs))

    def backward(self,dvalues):
        self.d_inputs = dvalues * (1-self.output**2)


class sigmoid:
    def forward(self,inputs):
        self.inputs = inputs
        self.output = 1/ (1+np.exp(-inputs))

    def backward(self,dvalues):
        self.d_inputs = dvalues * (self.output) * (1-self.output)


class softmax:
    def forward(self,inputs):
        self.inputs = inputs
        exp_values = np.exp(inputs - np.max(inputs,axis=1,keepdims=True))
        self.output = exp_values / np.sum(exp_values,axis = 1, keepdims= True)

    def backward(self,dvalues):
        diagonal = np.diag(self.output)[:None]
        pad = np.zeros(diagonal.shape)
        sii = np.hstack((np.diagflat(diagonal),pad))
        self.d_inputs = sii - (diagonal * self.output)
        #in progress, let skip it as it's not needed


class Loss:
    def forward(self,y_pred,y_true):
        self.batch_size = y_pred.shape[0]
        self.y_pred = y_pred
        self.y_true = y_true
        p = y_pred[y_true == 1]
        p = np.clip(p,0+1e-9,1 - 1e-9)
        self.loss =  - np.average(np.log(p))
    
    def backward(self):
        y_pred = np.clip(self.y_pred,0+1e-9,1 - 1e-9)
        self.d_loss =  - self.y_true/ (y_pred*self.batch_size)

    def backward_with_softmax(self):
        self.d_inputs = (self.y_pred - self.y_true) / self.batch_size


class optimizer:
    def __init__(self,lr = 0.001, beta = 0.9):
        self.lr = lr
        self.beta = beta

    def OptimizerGD(self,layer):
        layer.weights -= self.lr*layer.d_weights
        layer.biases -= self.lr*layer.d_biases

    def OptimizerGD_WithMomentumn(self, layer):
        
        layer.weight_momentumn = self.beta * layer.weight_momentumn - self.lr*layer.d_weights
        layer.biases_momentumn = self.beta * layer.biases_momentumn - self.lr*layer.d_biases

        layer.weights += layer.weight_momentumn
        layer.biases += layer.biases_momentumn

In [144]:
def OneHotMatrix(labels,num_classes):
    labels = np.asarray(labels, dtype=int).reshape(-1)
    y_true = np.zeros((len(labels), num_classes))
    y_true[np.arange(len(labels)), labels] = 1
    return y_true


def accuracy(y_pred, y_true):
    y_pred_labels = np.argmax(y_pred, axis=1)

    if len(y_true.shape) == 2:
        y_true_labels = np.argmax(y_true, axis=1)
    else:
        y_true_labels = y_true

    return np.mean(y_pred_labels == y_true_labels)

In [159]:
class Model:
    def __init__(self,layers):
        self.layers = layers

    def save(self,filename):
        with open (filename,'wb') as f:
            pickle.dump(self,f)

    @staticmethod
    def load(filename):
        with open(filename,'rb') as f:
            return pickle.load(f)
        
    def forward(self,X):
        output = X
        for layer in self.layers:
            layer.forward(output)
            output = layer.output
        return output

class Flatten:
    def forward(self, inputs):
        self.input_shape = inputs.shape
        self.output = inputs.reshape(inputs.shape[0], -1)

    def backward(self, dvalues):
        self.d_inputs = dvalues.reshape(self.input_shape)

data

In [146]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist.data, mnist.target
y = y.astype(np.int64)
X = X.astype(np.float32) / 255.0

X = X.reshape(-1, 28, 28, 1)

In [147]:
print(X.shape)  
print(y.shape)  

(70000, 28, 28, 1)
(70000,)


In [148]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42)



In [149]:
print(X_train.shape)
print(X_val.shape)

(44800, 28, 28, 1)
(11200, 28, 28, 1)


In [150]:
X_train = X_train[:4800]
y_train = y_train[:4800]
X_val = X_val[:1200]
y_val = y_val[:1200]

In [151]:
def after_conv_shape(img_width, img_height, kernel_size, n_filter):
    return (img_width - kernel_size + 1, img_height - kernel_size + 1, n_filter)

def after_pool_shape(img_width, img_height, channel, pool_size, stride ):
    return ( int(np.floor((img_width - pool_size)/stride) + 1), int(np.floor((img_height - pool_size)/stride) + 1), channel )

In [152]:
#CONFIG
epoches = 1000
batch_size = 64
channel_num = X_train.shape[3]
patience = 10

In [153]:
print(X_train.shape)

(4800, 28, 28, 1)


In [154]:
features_c1 = after_conv_shape(X_train.shape[1],X_train.shape[2],3,16)
features_p1 = after_pool_shape(*features_c1,2,2)

features_c2 = after_conv_shape(*features_p1[:-1],3,32)
features_p2 = after_pool_shape(*features_c2,2,2)


last_features_flattened = int(np.prod(features_p2))

In [155]:
print(features_p2)
print(last_features_flattened)

(5, 5, 32)
800


In [ ]:
#CONV

conv1 =  Convo(3, 16, channel_num)
conv_activation1 = relu()
pool1 = Pooling()

conv2 =  Convo(3, 32, 16)
conv_activation2 = relu()
pool2 = Pooling()



#MLP

ld1 = LayerDense(last_features_flattened, 16)
activation1 = relu()

ld2 = LayerDense(16, 10)
activation2 = softmax()

#LOST and OPTIMIZE

L = Loss()

opt = optimizer(lr=0.01)

conv_opt = conv_optimizer(lr=0.01)


flatten = Flatten()

In [ ]:
lowest_Loss = np.inf
num_classes = len(np.unique(y))
y_val_oh = OneHotMatrix(y_val,num_classes)
pat = 0
for epoch in range(epoches):

    epoch_loss = 0
    epoch_acc = 0
    num_batches = 0

    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]


    for b in range(0,len(X_train_shuffled),batch_size):
        X_batch = X_train_shuffled[b:b+batch_size]
        y_batch = OneHotMatrix(y_train_shuffled[b:b+batch_size],num_classes)

        conv1.forward(X_batch)
        conv_activation1.forward(conv1.feature_maps)
        pool1.forward(conv_activation1.output)

        conv2.forward(pool1.output)
        conv_activation2.forward(conv2.feature_maps)
        pool2.forward(conv_activation2.output)


        pool2_out = pool2.output
        flattened = pool2_out.reshape(pool2_out.shape[0],-1)

        
        ld1.forward(flattened)
        activation1.forward(ld1.output)

        ld2.forward(activation1.output)
        activation2.forward(ld2.output)

        L.forward(activation2.output,y_batch)


        acc = accuracy(activation2.output, y_batch)


        L.backward_with_softmax()

        ld2.backward(L.d_inputs)
        
        activation1.backward(ld2.d_inputs)
        ld1.backward(activation1.d_inputs)

        ld1_d = ld1.d_inputs
        restored = ld1_d.reshape(pool2_out.shape[0],*features_p2)

        pool2.backward(restored)
        conv_activation2.backward(pool2.d_inputs)
        conv2.backward(conv_activation2.d_inputs)

        pool1.backward(conv2.d_inputs)
        conv_activation1.backward(pool1.d_inputs)
        conv1.backward(conv_activation1.d_inputs)
        

        opt.OptimizerGD_WithMomentumn(ld2)
        opt.OptimizerGD_WithMomentumn(ld1)
        conv_opt.OptimizerGD_WithMomentumn(conv2)
        conv_opt.OptimizerGD_WithMomentumn(conv1)

        epoch_loss += L.loss
        epoch_acc += acc
        num_batches += 1


    epoch_avg_loss = epoch_loss / num_batches
    epoch_avg_acc = epoch_acc/num_batches

    #val

    conv1.forward(X_val)
    conv_activation1.forward(conv1.feature_maps)
    pool1.forward(conv_activation1.output)

    conv2.forward(pool1.output)
    conv_activation2.forward(conv2.feature_maps)
    pool2.forward(conv_activation2.output)


    pool2_out_val = pool2.output
    flattened_val = pool2_out_val.reshape(pool2_out_val.shape[0],-1)

    ld1.forward(flattened_val)
    activation1.forward(ld1.output)


    ld2.forward(activation1.output)
    activation2.forward(ld2.output)

    L.forward(activation2.output,y_val_oh)

    val_loss = L.loss

    val_acc = accuracy(activation2.output, y_val_oh)
    
    model = Model([conv1, conv_activation1,pool1, conv2,conv_activation2, pool2, flatten, ld1, activation1, ld2, activation2])

    if val_loss >= lowest_Loss:
        pat += 1
        if pat >= patience :
            break 
        
    else:
        pat = 0
        lowest_Loss = val_loss
        model.save('best_mnist_model.pkl')
        print('best model saved')

    

    print(f"Epoch {epoch+1}: train_acc: {epoch_avg_acc:.4f}, train_loss = {epoch_avg_loss:.4f}")
    print(f"Epoch {epoch+1}: val_acc: {val_acc:.4f}, val_loss = {val_loss:.4f}")





best model saved
Epoch 1: train_acc: 0.6215, train_loss = 1.1363
Epoch 1: val_acc: 0.8183, val_loss = 0.5575
best model saved
Epoch 2: train_acc: 0.8940, train_loss = 0.3418
Epoch 2: val_acc: 0.9217, val_loss = 0.2504
best model saved
Epoch 3: train_acc: 0.9256, train_loss = 0.2528
Epoch 3: val_acc: 0.9400, val_loss = 0.1999
best model saved
Epoch 4: train_acc: 0.9513, train_loss = 0.1618
Epoch 4: val_acc: 0.9383, val_loss = 0.1972
best model saved
Epoch 5: train_acc: 0.9606, train_loss = 0.1363
Epoch 5: val_acc: 0.9525, val_loss = 0.1527
best model saved
Epoch 6: train_acc: 0.9667, train_loss = 0.1090
Epoch 6: val_acc: 0.9558, val_loss = 0.1334
best model saved
Epoch 7: train_acc: 0.9773, train_loss = 0.0847
Epoch 7: val_acc: 0.9642, val_loss = 0.1317
Epoch 8: train_acc: 0.9779, train_loss = 0.0704
Epoch 8: val_acc: 0.9517, val_loss = 0.1552
best model saved
Epoch 9: train_acc: 0.9848, train_loss = 0.0549
Epoch 9: val_acc: 0.9658, val_loss = 0.1177
Epoch 10: train_acc: 0.9848, train_l

KeyboardInterrupt: 

In [ ]:
def predict_in_batches(model, X, batch_size=32):
    outputs = []
    for i in range(0, len(X), batch_size):
        X_batch = X[i:i+batch_size]
        out = model.forward(X_batch)
        outputs.append(out.copy())  
    return np.vstack(outputs)

In [ ]:
mnist_model = Model.load('best_mnist_model.pkl')
y_pred = predict_in_batches(mnist_model, X_test, batch_size=32)
mnist_acc = accuracy(y_pred, y_test)
print(f"test accuracy: {mnist_acc}")

test accuracy: 0.9617857142857142


In [167]:
print(f"test accuracy: {mnist_acc*100}%")

test accuracy: 96.17857142857143%
